Load Data

In [1]:
%run "../scripts/load_daily_variables.py"

Loading Variables: 100%|██████████| 26/26 [07:27<00:00, 17.21s/it]  


# Market Indicators

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# TC2000 definitions: https://help.tc2000.com/m/69404/c/213566


def _find_column(frame, candidates):
    """Return the first case-insensitive matching column name."""
    lookup = {str(column).casefold(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    return None


def _field_panel(field):
    """Align one OHLCV field from every SymbolData DataFrame by session."""
    series = {}
    for ticker, stock in tqdm(
        symbols.items(),
        total=len(symbols),
        desc=f"Loading {field} data",
        unit="symbol",
    ):
        frame = stock.df
        column = _find_column(frame, (field, field.capitalize(), field.upper()))
        if column is None or frame.empty:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        dates = pd.DatetimeIndex(pd.to_datetime(values.index, utc=True)).tz_localize(None).normalize()
        values = pd.Series(values.to_numpy(), index=dates, name=ticker)
        series[ticker] = values.groupby(level=0).last()
    if not series:
        raise ValueError(f"No {field!r} data was found in symbols[*].df")
    return pd.concat(series, axis=1).sort_index()


close = _field_panel("close")
high = _field_panel("high").reindex(index=close.index, columns=close.columns)
low = _field_panel("low").reindex(index=close.index, columns=close.columns)
volume = _field_panel("volume").reindex(index=close.index, columns=close.columns)

# Calculate every indicator over the complete set of symbols with price data.
all_symbols = list(close.columns)


def _breadth(universe):
    prices = close.reindex(columns=universe)
    previous = prices.shift(1)
    valid = prices.notna() & previous.notna()
    advancing = ((prices > previous) & valid).sum(axis=1)
    declining = ((prices < previous) & valid).sum(axis=1)
    unchanged = ((prices == previous) & valid).sum(axis=1)
    total = valid.sum(axis=1).replace(0, np.nan)
    advancing_volume = volume.reindex(columns=universe).where((prices > previous) & valid).sum(axis=1)
    declining_volume = volume.reindex(columns=universe).where((prices < previous) & valid).sum(axis=1)
    return pd.DataFrame(
        {
            "advancing": advancing,
            "declining": declining,
            "unchanged": unchanged,
            "total": total,
            "net_advances": advancing - declining,
            "advancing_volume": advancing_volume,
            "declining_volume": declining_volume,
        }
    )


def _advance_decline_line(universe):
    breadth = _breadth(universe)
    daily_value = 100 * breadth["net_advances"].div(breadth["total"])
    return daily_value.fillna(0).cumsum().rename("advance_decline_line")


def _new_high_low_counts(universe, sessions):
    universe_high = high.reindex(columns=universe)
    universe_low = low.reindex(columns=universe)
    rolling_high = universe_high.rolling(sessions, min_periods=sessions).max()
    rolling_low = universe_low.rolling(sessions, min_periods=sessions).min()
    new_highs = (universe_high >= rolling_high).sum(axis=1)
    new_lows = (universe_low <= rolling_low).sum(axis=1)
    return new_highs, new_lows


def _new_high_low_ratio(sessions):
    new_highs, new_lows = _new_high_low_counts(all_symbols, sessions)
    denominator = (new_highs + new_lows).replace(0, np.nan)
    return (100 * new_highs.div(denominator)).rename("new_high_new_low_ratio")


def _percentage_relative_to_pma(period, channels=0, direction="above"):
    prices = close.reindex(columns=all_symbols)
    pma = prices.rolling(period, min_periods=period).mean()
    deviation = prices.rolling(period, min_periods=period).std(ddof=0)
    boundary = pma + channels * deviation if direction == "above" else pma - channels * deviation
    eligible = prices.notna() & boundary.notna()
    matches = (prices > boundary) if direction == "above" else (prices < boundary)
    return (100 * (matches & eligible).sum(axis=1).div(eligible.sum(axis=1).replace(0, np.nan)))


breadth = _breadth(all_symbols)

# T2100, T2125-T2129: cumulative advancing percentage minus declining percentage.
advance_decline_line = _advance_decline_line(all_symbols)

# T2101: absolute five-session net advances as a percentage of all issues.
absolute_breadth_index = (
    100
    * breadth["net_advances"].rolling(5, min_periods=5).sum().abs()
    .div(breadth["total"])
).rename("absolute_breadth_index")

# T2102: cumulative signed square root of (advances/unchanged - declines/unchanged).
bolton_tremblay_increment = np.sign(breadth["net_advances"]) * np.sqrt(
    breadth["net_advances"].abs().div(breadth["unchanged"].replace(0, np.nan))
)
bolton_tremblay_indicator = bolton_tremblay_increment.fillna(0).cumsum().rename(
    "bolton_tremblay_indicator"
)

# T2103: ten-session SMA of advances/(advances + declines), on a 0-100 scale.
zweig_breadth_thrust = (
    100
    * breadth["advancing"]
    .div((breadth["advancing"] + breadth["declining"]).replace(0, np.nan))
    .rolling(10, min_periods=10)
    .mean()
).rename("zweig_breadth_thrust")

# T2104: cumulative advancing-issue volume minus declining-issue volume.
cumulative_volume_index = (
    breadth["advancing_volume"] - breadth["declining_volume"]
).fillna(0).cumsum().rename("cumulative_volume_index")

# T2105 uses 52-week (260-session) new highs/lows and the smaller percentage.
new_highs_52_week, new_lows_52_week = _new_high_low_counts(all_symbols, 260)
high_low_logic_index = (
    100
    * pd.concat([new_highs_52_week, new_lows_52_week], axis=1).min(axis=1)
    .div(breadth["total"])
).rename("high_low_logic_index")

# T2106 and T2118: 19-day EMA - 39-day EMA of net advances, then cumulative sum.
mcclellan_oscillator = (
    breadth["net_advances"].ewm(span=19, adjust=False).mean()
    - breadth["net_advances"].ewm(span=39, adjust=False).mean()
).rename("mcclellan_oscillator")
mcclellan_summation_index = mcclellan_oscillator.fillna(0).cumsum().rename(
    "mcclellan_summation_index"
)

# T2117, T2120-T2122: new highs / (new highs + new lows), on a 0-100 scale.
new_high_new_low_ratio_52_week = _new_high_low_ratio(260)
new_high_new_low_ratio_26_week = _new_high_low_ratio(130)
new_high_new_low_ratio_13_week = _new_high_low_ratio(65)
new_high_new_low_ratio_4_week = _new_high_low_ratio(20)

# T2123's historical name says cumulative, but TC2000 defines it as today's
# count of four-week new highs minus today's count of four-week new lows.
new_highs_4_week, new_lows_4_week = _new_high_low_counts(all_symbols, 20)
cumulative_4_week_new_high_low = (new_highs_4_week - new_lows_4_week).rename(
    "cumulative_4_week_new_high_low"
)

# T2107-T2116: percentages relative to simple PMAs and population std-dev channels.
percentage_of_stocks_above_200_day_pma = _percentage_relative_to_pma(200)
percentage_of_stocks_above_40_day_pma = _percentage_relative_to_pma(40)
percentage_of_stocks_1_channel_above_200_day_pma = _percentage_relative_to_pma(200, 1, "above")
percentage_of_stocks_1_channel_above_40_day_pma = _percentage_relative_to_pma(40, 1, "above")
percentage_of_stocks_1_channel_below_200_day_pma = _percentage_relative_to_pma(200, 1, "below")
percentage_of_stocks_1_channel_below_40_day_pma = _percentage_relative_to_pma(40, 1, "below")
percentage_of_stocks_2_channels_above_200_day_pma = _percentage_relative_to_pma(200, 2, "above")
percentage_of_stocks_2_channels_above_40_day_pma = _percentage_relative_to_pma(40, 2, "above")
percentage_of_stocks_2_channels_below_200_day_pma = _percentage_relative_to_pma(200, 2, "below")
percentage_of_stocks_2_channels_below_40_day_pma = _percentage_relative_to_pma(40, 2, "below")

# One aligned table is convenient for comparison while the requested text-name
# variables above remain directly available as Series.
tc2000_t2_indicators = pd.DataFrame(
    {
        "absolute_breadth_index": absolute_breadth_index,
        "advance_decline_line": advance_decline_line,
        "bolton_tremblay_indicator": bolton_tremblay_indicator,
        "cumulative_4_week_new_high_low": cumulative_4_week_new_high_low,
        "cumulative_volume_index": cumulative_volume_index,
        "high_low_logic_index": high_low_logic_index,
        "mcclellan_oscillator": mcclellan_oscillator,
        "mcclellan_summation_index": mcclellan_summation_index,
        "new_high_new_low_ratio_4_week": new_high_new_low_ratio_4_week,
        "new_high_new_low_ratio_13_week": new_high_new_low_ratio_13_week,
        "new_high_new_low_ratio_26_week": new_high_new_low_ratio_26_week,
        "new_high_new_low_ratio_52_week": new_high_new_low_ratio_52_week,
        "percentage_of_stocks_1_channel_above_200_day_pma": percentage_of_stocks_1_channel_above_200_day_pma,
        "percentage_of_stocks_1_channel_above_40_day_pma": percentage_of_stocks_1_channel_above_40_day_pma,
        "percentage_of_stocks_1_channel_below_200_day_pma": percentage_of_stocks_1_channel_below_200_day_pma,
        "percentage_of_stocks_1_channel_below_40_day_pma": percentage_of_stocks_1_channel_below_40_day_pma,
        "percentage_of_stocks_2_channels_above_200_day_pma": percentage_of_stocks_2_channels_above_200_day_pma,
        "percentage_of_stocks_2_channels_above_40_day_pma": percentage_of_stocks_2_channels_above_40_day_pma,
        "percentage_of_stocks_2_channels_below_200_day_pma": percentage_of_stocks_2_channels_below_200_day_pma,
        "percentage_of_stocks_2_channels_below_40_day_pma": percentage_of_stocks_2_channels_below_40_day_pma,
        "percentage_of_stocks_above_200_day_pma": percentage_of_stocks_above_200_day_pma,
        "percentage_of_stocks_above_40_day_pma": percentage_of_stocks_above_40_day_pma,
        "zweig_breadth_thrust": zweig_breadth_thrust,
    }
)

tc2000_t2_indicators.tail()

Loading volume data: 100%|██████████| 3037/3037 [00:23<00:00, 126.59symbol/s]


,absolute_breadth_index,advance_decline_line,bolton_tremblay_indicator,cumulative_4_week_new_high_low,cumulative_volume_index,high_low_logic_index,mcclellan_oscillator,mcclellan_summation_index,new_high_new_low_ratio_4_week,new_high_new_low_ratio_13_week,new_high_new_low_ratio_26_week,new_high_new_low_ratio_52_week,percentage_of_stocks_1_channel_above_200_day_pma,percentage_of_stocks_1_channel_above_40_day_pma,percentage_of_stocks_1_channel_below_200_day_pma,percentage_of_stocks_1_channel_below_40_day_pma,percentage_of_stocks_2_channels_above_200_day_pma,percentage_of_stocks_2_channels_above_40_day_pma,percentage_of_stocks_2_channels_below_200_day_pma,percentage_of_stocks_2_channels_below_40_day_pma,percentage_of_stocks_above_200_day_pma,percentage_of_stocks_above_40_day_pma,zweig_breadth_thrust
Date,,,,,,,,,,,,,,,,,,,,,,,
2026-07-22,44.944945,9742.436803,703.871752,119,9.000725e+11,0.667334,-65.228815,876.968611,63.370787,66.867470,74.576271,78.947368,37.478530,25.150301,17.107523,17.234469,9.446925,3.740815,1.305393,1.469606,61.147372,57.047428,50.283893
2026-07-23,92.559226,9709.804170,697.350883,-218,8.953934e+11,1.968635,-109.055480,767.913132,34.428571,35.627530,38.414634,45.384615,36.551013,21.175685,19.134318,21.843687,8.175885,3.173013,1.751975,2.972612,59.292339,52.204409,46.934868
2026-07-24,46.128171,9723.989751,701.179090,-48,8.943727e+11,1.535381,-75.832001,692.081131,45.774648,52.614379,58.937198,70.322581,36.735395,25.993986,18.350515,22.753091,9.690722,3.575008,1.512027,2.940194,60.103093,54.594053,47.266465
2026-07-27,20.260347,9755.231407,706.764786,103,8.978698e+11,0.500668,-21.496766,670.584364,58.034321,66.666667,78.109453,90.066225,36.713647,31.216578,15.847370,19.184492,10.828463,4.278075,1.271915,2.105615,62.255070,58.823529,49.441513
2026-07-28,16.421896,9777.594557,712.048409,229,8.987243e+11,1.034713,11.767343,682.351707,59.256265,67.283073,75.742574,88.301887,38.131226,38.636364,14.702851,20.354278,13.740982,11.898396,1.374098,4.044118,62.796290,60.561497,50.263060


# Color Scheme Summary
## median
### 1 abs deviation
- abosolute_breadth_index
- new_high_new_low_ratio_4_week
- new_high_new_low_ratio_13_week
- new_high_new_low_ratio_26_week
- new_high_new_low_ratio_52_week
- percentage_of_stocks_2_channels_above_200_day_pma
- percentage_of_stocks_2_channels_below_200_day_pma
- percentage_of_stocks_2_channels_above_40_day_pma
- percentage_of_stocks_2_channels_below_40_day_pma
#### 2 years
- high_low_logic_index
### 2 abs deviation
- percentage_of_stocks_above_200_day_pma
- percentage_of_stocks_1_channel_above_200_day_pma
- percentage_of_stocks_1_channel_below_200_day_pma
- percentage_of_stocks_above_40_day_pma
- percentage_of_stocks_1_channel_above_40_day_pma
- percentage_of_stocks_1_channel_below_40_day_pma
#### 2 years
- cumulative_4_week_new_high_low
- mcclellan_summation_index
## increasing/decreasing
- advance_decline_line
- bolton_tremblay_indicator
- cumulative_volume_index
## mean
### 2 deviation
- zweig_breadth_thrust
#### 2 years
- mcclellan_oscillator


In [147]:
from scipy.stats import median_abs_deviation

# Green denotes values above the center; red denotes values below it.
# The shade darkens as the value moves farther from the center.
_GREEN = ("#d9ead3", "#93c47d", "#38761d")
_RED = ("#f4cccc", "#e06666", "#990000")

_MEDIAN_1_MAD = {
    "absolute_breadth_index",
    "new_high_new_low_ratio_4_week",
    "new_high_new_low_ratio_13_week",
    "new_high_new_low_ratio_26_week",
    "new_high_new_low_ratio_52_week",
    "percentage_of_stocks_2_channels_above_200_day_pma",
    "percentage_of_stocks_2_channels_below_200_day_pma",
    "percentage_of_stocks_2_channels_above_40_day_pma",
    "percentage_of_stocks_2_channels_below_40_day_pma",
}
_MEDIAN_1_MAD_2_YEARS = {"high_low_logic_index"}

_MEDIAN_2_MAD = {
    "percentage_of_stocks_above_200_day_pma",
    "percentage_of_stocks_1_channel_above_200_day_pma",
    "percentage_of_stocks_1_channel_below_200_day_pma",
    "percentage_of_stocks_above_40_day_pma",
    "percentage_of_stocks_1_channel_above_40_day_pma",
    "percentage_of_stocks_1_channel_below_40_day_pma",
}
_MEDIAN_2_MAD_2_YEARS = {
    "cumulative_4_week_new_high_low",
    "mcclellan_summation_index",
}

_INCREASING_DECREASING = {
    "advance_decline_line",
    "bolton_tremblay_indicator",
    "cumulative_volume_index",
}

_MEAN_2_STD = {"zweig_breadth_thrust"}
_MEAN_2_STD_2_YEARS = {"mcclellan_oscillator"}


def _background_css(color, darkest=False):
    text_color = "; color: white" if darkest else ""
    return f"background-color: {color}{text_color}"


def _centered_column_styles(values, center, deviation, two_deviation_shades):
    styles = pd.Series("", index=values.index, dtype="object")
    valid = values.notna() & pd.notna(center)

    distance = (values - center).abs()
    if not pd.notna(deviation) or deviation <= 0:
        shade = pd.Series(0, index=values.index)
    elif two_deviation_shades:
        # Light through 1 deviation, medium through 2, then dark.
        shade = (distance.gt(deviation).astype(int) + distance.gt(2 * deviation).astype(int))
    else:
        # Light through 1 absolute deviation, then dark.
        shade = distance.gt(deviation).astype(int) * 2

    for direction, palette in ((values > center, _GREEN), (values < center, _RED)):
        for level, color in enumerate(palette):
            mask = valid & direction & shade.eq(level)
            styles.loc[mask] = _background_css(color, darkest=(level == 2))
    return styles


def _trend_column_styles(values):
    change_direction = np.sign(values.diff()).fillna(0)
    run_group = change_direction.ne(change_direction.shift()).cumsum()
    run_length = change_direction.groupby(run_group).cumcount().add(1).clip(upper=3)
    styles = pd.Series("", index=values.index, dtype="object")

    for direction, palette in ((1, _GREEN), (-1, _RED)):
        for run_day, color in enumerate(palette, start=1):
            mask = change_direction.eq(direction) & run_length.eq(run_day)
            styles.loc[mask] = _background_css(color, darkest=(run_day == 3))
    return styles


def _tc2000_t2_style_matrix(frame):
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    latest_date = pd.Timestamp(frame.index.max())
    two_year_start = latest_date - pd.DateOffset(years=2)

    median_groups = (
        (_MEDIAN_1_MAD, False, False),
        (_MEDIAN_1_MAD_2_YEARS, True, False),
        (_MEDIAN_2_MAD, False, True),
        (_MEDIAN_2_MAD_2_YEARS, True, True),
    )
    for columns, use_two_years, two_deviation_shades in median_groups:
        for column in columns:
            history = frame.loc[frame.index >= two_year_start, column] if use_two_years else frame[column]
            center = history.median()
            deviation = median_abs_deviation(history, nan_policy="omit")
            styles[column] = _centered_column_styles(
                frame[column], center, deviation, two_deviation_shades
            )

    mean_groups = ((_MEAN_2_STD, False), (_MEAN_2_STD_2_YEARS, True))
    for columns, use_two_years in mean_groups:
        for column in columns:
            history = frame.loc[frame.index >= two_year_start, column] if use_two_years else frame[column]
            center = history.mean()
            two_std = history.std() * 2
            styles[column] = _centered_column_styles(
                frame[column], center, two_std / 2, True
            )

    for column in _INCREASING_DECREASING:
        styles[column] = _trend_column_styles(frame[column])

    return styles


def style_tc2000_t2_indicators(frame, rows=None):
    """Return a styled view; use rows to limit notebook rendering to recent sessions."""
    styles = _tc2000_t2_style_matrix(frame)
    display_frame = frame if rows is None else frame.tail(rows)
    display_styles = styles.loc[display_frame.index]
    return display_frame.style.apply(lambda _: display_styles, axis=None).format(precision=2)


# Render recent sessions without building a very large HTML table in the notebook.
tc2000_t2_indicators_styled = style_tc2000_t2_indicators(tc2000_t2_indicators, rows=10)
tc2000_t2_indicators_styled

,absolute_breadth_index,advance_decline_line,bolton_tremblay_indicator,cumulative_4_week_new_high_low,cumulative_volume_index,high_low_logic_index,mcclellan_oscillator,mcclellan_summation_index,new_high_new_low_ratio_4_week,new_high_new_low_ratio_13_week,new_high_new_low_ratio_26_week,new_high_new_low_ratio_52_week,percentage_of_stocks_1_channel_above_200_day_pma,percentage_of_stocks_1_channel_above_40_day_pma,percentage_of_stocks_1_channel_below_200_day_pma,percentage_of_stocks_1_channel_below_40_day_pma,percentage_of_stocks_2_channels_above_200_day_pma,percentage_of_stocks_2_channels_above_40_day_pma,percentage_of_stocks_2_channels_below_200_day_pma,percentage_of_stocks_2_channels_below_40_day_pma,percentage_of_stocks_above_200_day_pma,percentage_of_stocks_above_40_day_pma,zweig_breadth_thrust
Date,,,,,,,,,,,,,,,,,,,,,,,
2026-07-15 00:00:00,50.58,9787.38,710.70,165,903740274611.55,1.03,11.05,1069.34,62.33,73.22,73.30,80.62,38.65,33.81,15.80,17.86,12.64,5.21,1.10,1.13,62.18,61.38,51.20
2026-07-16 00:00:00,31.75,9802.35,715.56,315,901814833257.46,1.07,26.55,1095.89,64.54,70.51,75.37,89.15,40.10,40.67,15.40,19.20,15.84,12.69,1.20,3.37,63.47,63.87,51.68
2026-07-17 00:00:00,8.11,9770.10,708.78,57,897717552281.61,1.33,-31.13,1064.76,52.62,56.23,64.74,82.38,37.80,35.63,16.84,21.44,14.12,6.44,1.48,3.24,61.41,60.20,49.37
2026-07-20 00:00:00,31.43,9734.96,702.85,-23,897108332213.56,1.20,-84.59,980.17,47.60,50.21,56.63,68.70,36.04,30.26,17.66,23.98,11.34,3.91,1.34,2.87,60.29,56.35,47.21
2026-07-21 00:00:00,11.28,9761.16,708.35,24,901034044988.26,0.93,-37.97,942.20,52.64,61.38,63.50,72.28,38.10,29.79,16.59,17.64,11.95,4.04,1.37,1.10,61.18,58.72,49.25
2026-07-22 00:00:00,44.94,9742.44,703.87,119,900072526700.49,0.67,-65.23,876.97,63.37,66.87,74.58,78.95,37.48,25.15,17.11,17.23,9.45,3.74,1.31,1.47,61.15,57.05,50.28
2026-07-23 00:00:00,92.56,9709.80,697.35,-218,895393429976.41,1.97,-109.06,767.91,34.43,35.63,38.41,45.38,36.55,21.18,19.13,21.84,8.18,3.17,1.75,2.97,59.29,52.20,46.93
2026-07-24 00:00:00,46.13,9723.99,701.18,-48,894372716983.54,1.54,-75.83,692.08,45.77,52.61,58.94,70.32,36.74,25.99,18.35,22.75,9.69,3.58,1.51,2.94,60.10,54.59,47.27
2026-07-27 00:00:00,20.26,9755.23,706.76,103,897869790253.80,0.50,-21.50,670.58,58.03,66.67,78.11,90.07,36.71,31.22,15.85,19.18,10.83,4.28,1.27,2.11,62.26,58.82,49.44
